# 002 · Class and Static Members

> **Syntax › Classes & OOP** · Python 3.14

C# has one keyword — `static` — for members that belong to the type rather than an instance. Python splits this idea into **class attributes**, **`@staticmethod`** and **`@classmethod`**, and adds a twist C# does not have: class attributes can be *shadowed* by instance attributes.

**In this notebook you will learn:**

- class attributes and how they compare to `static` fields
- the shadowing and shared-mutable-state gotchas of class attributes
- `@staticmethod` vs `@classmethod` (and what `cls` is)
- alternative constructors with `@classmethod` (the Python take on static factory methods)
- why module-level functions often replace C# static utility classes

## C# mental model

| C# | Python |
|---|---|
| `static int Count;` | class attribute: `count = 0` in the class body |
| `static void Helper()` | `@staticmethod def helper(): ...` |
| `static Point FromPolar(...)` (factory) | `@classmethod def from_polar(cls, ...): ...` |
| `static class MathUtils { ... }` | a **module** with plain functions |
| `const` / `static readonly` | class attribute in `UPPER_CASE` (convention only) |

The key mental shift: a class attribute is a **default shared by all instances** that is looked up *through* the instance when the instance has no attribute of that name. It is not a separate, strictly static storage slot.

## Class attributes vs instance attributes

Any variable assigned directly in the class body (not inside a method) is a **class attribute**, shared by every instance. Attributes assigned on `self` are **instance attributes**.

```csharp
// C#
public class Product
{
    public static decimal TaxRate = 0.2m;   // shared
    public string Name { get; }              // per instance
    public Product(string name) => Name = name;
}
```

In [ ]:
class Product:
    tax_rate = 0.2                   # class attribute — shared

    def __init__(self, name, price):
        self.name = name             # instance attributes
        self.price = price

    def price_with_tax(self):
        return self.price * (1 + self.tax_rate)   # found on the class via self

pen = Product("pen", 10)
book = Product("book", 20)
print(Product.tax_rate, pen.tax_rate, book.tax_rate)   # all 0.2
print(pen.price_with_tax())

Unlike C#, a class attribute can be read **through an instance** (`pen.tax_rate`) as well as through the class (`Product.tax_rate`). Attribute lookup checks the instance's `__dict__` first, then the class, then base classes.

Changing it on the class affects every instance:

In [ ]:
Product.tax_rate = 0.25
print(pen.price_with_tax(), book.price_with_tax())
print(pen.__dict__)          # tax_rate is NOT stored on the instance
print("tax_rate" in Product.__dict__)

## ⚠️ Gotcha: assigning through an instance shadows the class attribute

Assigning `instance.attr = value` **always** writes to the instance — it creates a new instance attribute that hides (shadows) the class attribute for that one object. The class attribute itself is unchanged.

In [ ]:
pen.tax_rate = 0.0           # creates an INSTANCE attribute on pen only

print(pen.tax_rate)          # 0.0  — instance attribute wins
print(book.tax_rate)         # 0.25 — still sees the class attribute
print(Product.tax_rate)      # 0.25 — class attribute untouched
print(pen.__dict__)

del pen.tax_rate             # remove the shadow...
print(pen.tax_rate)          # ...and the class value shows through again

So to change shared state from inside a method, write to the class explicitly (`Product.count += 1` or `type(self).count += 1`) — **not** `self.count += 1`, which reads the class value but writes an instance attribute.

## Example: counting instances

```csharp
// C#
public class Order
{
    public static int Created { get; private set; }
    public Order() => Created++;
}
```

In [ ]:
class Order:
    created = 0                       # class-level counter

    def __init__(self, item):
        self.item = item
        Order.created += 1            # write to the CLASS, not self

for item in ["pen", "book", "lamp"]:
    Order(item)

print(Order.created)    # 3

The buggy version — `self.created += 1` — silently gives every instance its own counter:

In [ ]:
class BuggyOrder:
    created = 0

    def __init__(self):
        self.created += 1             # reads class (0), writes instance attr (1)

a, b = BuggyOrder(), BuggyOrder()
print(a.created, b.created, BuggyOrder.created)   # 1 1 0 — the class never changed

## ⚠️ Gotcha: mutable class attributes are shared state

The opposite problem: if a class attribute holds a **mutable** object (list, dict, set) and you *mutate* it through `self`, no shadowing happens — you are modifying the single shared object. This is a very common bug for people coming from C#, where a field initialiser like `List<string> Tags = new();` runs **per instance**.

```csharp
// C# — each Cart gets its own list
public class Cart { public List<string> Items { get; } = new(); }
```

In [ ]:
class Cart:
    items = []                        # ONE list, shared by all carts!

    def add(self, item):
        self.items.append(item)       # mutates the shared list

cart1, cart2 = Cart(), Cart()
cart1.add("apple")
print(cart2.items)                    # ['apple'] — surprise!

The fix: create per-instance mutable state in `__init__`.

In [ ]:
class Cart:
    def __init__(self):
        self.items = []               # a new list for every instance

    def add(self, item):
        self.items.append(item)

cart1, cart2 = Cart(), Cart()
cart1.add("apple")
print(cart1.items, cart2.items)       # ['apple'] []

Rule of thumb: class attributes are great for **immutable** shared values (constants, defaults, counters you reassign). Mutable per-object data belongs in `__init__`.

## Constants

Python has no `const` or `static readonly`. By convention, `UPPER_CASE` names signal "do not reassign". Type checkers honour `typing.Final`, but nothing is enforced at runtime.

In [ ]:
from typing import Final

class Circle:
    PI: Final = 3.14159               # C#: public const double Pi = 3.14159;

    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return Circle.PI * self.radius ** 2

print(Circle(2).area())

## `@staticmethod` — a plain function in the class namespace

A static method receives **neither** the instance nor the class. It is just a function that lives inside the class for organisational reasons — exactly like a C# `static` method that does not touch any static state.

```csharp
// C#
public class Temperature
{
    public static double CToF(double c) => c * 9 / 5 + 32;
}
Temperature.CToF(100);
```

In [ ]:
class Temperature:
    @staticmethod
    def c_to_f(c):
        return c * 9 / 5 + 32

print(Temperature.c_to_f(100))       # call on the class
print(Temperature().c_to_f(0))       # also callable via an instance

## `@classmethod` — receives the class as `cls`

A class method receives the **class itself** as its first argument, conventionally named `cls` (just as instance methods receive `self`). C# has no direct equivalent: a C# static method knows its declaring type at compile time, whereas `cls` is the class it was **actually called on** — which matters for inheritance.

In [ ]:
class Animal:
    sound = "..."

    @classmethod
    def describe(cls):
        return f"{cls.__name__} says {cls.sound}"

class Dog(Animal):
    sound = "woof"

print(Animal.describe())   # Animal says ...
print(Dog.describe())      # Dog says woof — cls is Dog here

## Alternative constructors with `@classmethod`

Since a class has only one `__init__`, Python uses class methods as **named alternative constructors**. This is the idiomatic replacement for C# constructor overloads and static factory methods. Standard-library examples include `dict.fromkeys`, `datetime.fromisoformat` and `int.from_bytes`.

```csharp
// C#
public record Point(double X, double Y)
{
    public static Point FromPolar(double r, double theta) =>
        new(r * Math.Cos(theta), r * Math.Sin(theta));

    public static Point Parse(string s) { ... }
}
```

In [ ]:
import math

class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    @classmethod
    def from_polar(cls, r, theta):
        return cls(r * math.cos(theta), r * math.sin(theta))   # cls(...), not Point(...)

    @classmethod
    def parse(cls, text):
        x, y = text.split(",")
        return cls(float(x), float(y))

    def __repr__(self):
        return f"{type(self).__name__}({self.x:.2f}, {self.y:.2f})"

print(Point.from_polar(1, math.pi / 2))
print(Point.parse("3, 4"))

Because the factory calls `cls(...)` rather than hard-coding `Point(...)`, it automatically produces the right type for subclasses — something a C# static factory would need generics or overriding for:

In [ ]:
class Point3D(Point):
    pass

p = Point3D.parse("1, 2")
print(p, type(p).__name__)    # Point3D — the factory is inherited and still correct

## Which one should I use?

| You need… | Use |
|---|---|
| access to the instance (`self`) | regular method |
| access to the class (`cls`), e.g. factories, class-level state | `@classmethod` |
| neither — a helper that logically belongs to the class | `@staticmethod` |
| neither, and it doesn't really belong to the class | a **module-level function** |

## Module-level functions instead of static utility classes

In C#, every function must live in a class, which leads to `static class StringUtils { ... }`. Python modules are themselves namespaces, so utility functions usually go **directly in a module** — no class needed. You then call them as `text_utils.slugify(...)` after `import text_utils`, which reads just like `StringUtils.Slugify(...)`.

```csharp
// C#
public static class TextUtils
{
    public static string Slugify(string s) => s.Trim().ToLower().Replace(' ', '-');
}
```

In [ ]:
# In a real project this would live in text_utils.py
def slugify(text):
    return text.strip().lower().replace(" ", "-")

print(slugify("  Hello World "))

A class made only of static methods is considered un-Pythonic. Reach for `@staticmethod` only when the function is tightly related to the class (e.g. a validation helper used by its methods). Modules are covered in *Modules & Packages › 002 · Creating Modules and Packages*.

## Summary

| Concept | C# | Python |
|---|---|---|
| Shared field | `static int Count;` | class attribute `count = 0` |
| Read shared field | `Type.Count` | `Type.count` (or `self.count` to read) |
| Write shared field | `Count++` | `Type.count += 1` (never `self.count += 1`) |
| Per-instance collection | `List<T> Items = new();` | `self.items = []` in `__init__` |
| Constant | `const` / `static readonly` | `UPPER_CASE` + optional `typing.Final` |
| Static helper | `static` method | `@staticmethod` |
| Method receiving the type | (none) | `@classmethod` with `cls` |
| Factory / extra constructor | static factory method / overload | `@classmethod` returning `cls(...)` |
| Static utility class | `static class Utils` | module-level functions |

## Exercises

1. Write a `Ticket` class that assigns each new instance a unique, increasing `id` (1, 2, 3, …) using a class attribute.
2. Add a `@classmethod` `from_dict(cls, data)` to a `User` class with `name` and `age`, so that `User.from_dict({"name": "alice", "age": 30})` works.
3. Find the bug and fix it:
   ```python
   class Team:
       members = []
       def join(self, name):
           self.members.append(name)
   ```
4. Without running it, predict the output: `class A: x = 1` then `a = A(); a.x = 2; A.x = 3; print(a.x, A().x)`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
class Ticket:
    _next_id = 1

    def __init__(self):
        self.id = Ticket._next_id
        Ticket._next_id += 1

print([Ticket().id for _ in range(3)])   # [1, 2, 3]

# 2
class User:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    @classmethod
    def from_dict(cls, data):
        return cls(data["name"], data["age"])

u = User.from_dict({"name": "alice", "age": 30})

# 3 — `members` is one list shared by every Team; create it per instance
class Team:
    def __init__(self):
        self.members = []

    def join(self, name):
        self.members.append(name)

# 4 — prints "2 3": a.x is an instance attribute that shadows A.x;
#     a brand-new A() has no instance attribute and sees the class value 3.
```

</details>